# Study 883 — Mid-Cap Sweet Spot 🎯

**Are mid-caps the "forgotten middle" — a better risk-adjusted return than BOTH large
(SPY) and small (IWM)?**

Folklore (and a lot of ETF marketing) says the mid-cap band is the sweet spot: past
small-cap fragility, still growing faster than mega-caps, and under-followed by analysts.
We test whether the mid-cap ETF (**IJH**, plus **MDY** for the longer S&P MidCap 400 tape)
delivers a genuine **excess-of-cash Sharpe advantage over BOTH** neighbours
(2007-05 -> 2026-06, cash = BIL), whether it holds across eras, and whether it survives costs.

*Numbers below are the frozen headline (`docs/results.md`, fingerprint
`d294f0cdb517`); the live cells run the fast synthetic control. Short cash history:
BIL lists only from 2007, so the Sharpe race misses the 1995-2006 mid heyday — named on
the Signal axis.*


## 1. The idea in one picture

Small-caps promise growth but carry fragility, illiquidity and default risk. Mega-caps are safe but slow and picked over by every analyst alive. The **middle** — big enough to be stable, small enough to still compound, and under-covered — is supposed to be where the best *risk-adjusted* return hides. If true, the mid-cap ETF should out-Sharpe **both** SPY and IWM.

In [1]:
R = dict(ijh_sh=0.453, spy_sh=0.542, iwm_sh=0.394)
print('excess-of-cash Sharpe, 2007-2026 (higher = better):')
print(f"  large SPY : {R['spy_sh']:.3f}")
print(f"  MID  IJH : {R['ijh_sh']:.3f}   <- the 'sweet spot'")
print(f"  small IWM : {R['iwm_sh']:.3f}")
print('mid is literally in the MIDDLE - below large, just above small.')

excess-of-cash Sharpe, 2007-2026 (higher = better):
  large SPY : 0.542
  MID  IJH : 0.453   <- the 'sweet spot'
  small IWM : 0.394
mid is literally in the MIDDLE - below large, just above small.


## 2. So did the middle win? No — it sat in the middle

On the excess-of-cash Sharpe race mid-cap **fails the 'beats BOTH' test on its own terms**: IJH's 0.453 is *below* large SPY's 0.542 and only just above small IWM's 0.394. The paired bootstrap says the advantage is not distinguishable from zero either way — vs SPY **-0.089** (95% CI [-0.225, +0.050]), vs IWM **+0.060** (CI [-0.038, +0.169]). The 2010s mega-cap tech run handed the best Sharpe to *large*, not the middle.

## 3. But mid *did* out-return both over the long run — just not reliably

The return **difference** doesn't need the cash leg, so it reaches back to 1995. Over the full tape mid-cap out-returned both: **MDY − SPY = +0.95%/yr** (1995-2026), **IJH − SPY = +1.74%/yr**, IJH − IWM = +0.44%/yr. The *sign* is real — the forgotten middle tilt exists — but **not one difference clears a HAC *t* of 2** (best is +1.19 over 26 years). And it is not stable:

In [2]:
R = dict(era1_d=3.21, era1_t=0.94, era4_d=-3.5, era4_t=-1.23)
print('MDY - SPY by era (+ = mid beats large):')
print(f"  1995-2002 : {R['era1_d']:+.2f}%/yr  (HAC t {R['era1_t']:+.2f})")
print(f"  2017-2026 : {R['era4_d']:+.2f}%/yr  (HAC t {R['era4_t']:+.2f})  <- REVERSED")

MDY - SPY by era (+ = mid beats large):
  1995-2002 : +3.21%/yr  (HAC t +0.94)
  2017-2026 : -3.50%/yr  (HAC t -1.23)  <- REVERSED


## 4. A live synthetic control — the detector works

To be sure the 'no robust advantage' reading isn't a dead detector, we plant a real mid Sharpe edge in a seeded toy world and check it fires — and stays quiet on the null. No network.

In [3]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..','..','..')))
from midcap import data, strategy as st
null = st.synthetic_detect(data.synthetic_world(n_days=3000, edge=0.0, seed=883))
planted = st.synthetic_detect(data.synthetic_world(n_days=3000, edge=0.0006, seed=883))
print('null world   : mid adv vs large = %+.3f (t %+.2f) -> beats_both=%s'
      % (null['adv_large'], null['t_large'], null['beats_both']))
print('planted world: mid adv vs large = %+.3f (t %+.2f) -> beats_both=%s'
      % (planted['adv_large'], planted['t_large'], planted['beats_both']))

null world   : mid adv vs large = +0.091 (t +0.46) -> beats_both=True
planted world: mid adv vs large = +1.011 (t +5.78) -> beats_both=True


## 5. The honest verdict

- **Signal: Weak.** Mid-cap is real folklore with a real *sign* — it out-returned both neighbours over the long run — but the advantage **never reaches significance, sits below large on a modern Sharpe basis, and reversed in the last decade** (-3.50%/yr vs large in 2017-2026). A fragile, era-dependent tilt, not a dependable sweet spot.
- **Tradability: Mirage.** Long-mid/short-large nets only **+1.00%/yr** (net *t* = +0.68, and on the leg that inverted); long-mid/short-small nets **-0.30%/yr** after costs. Nothing bankable.